In [19]:
import pandas as pd

TARGET_PATH = './data/data.csv'

df = pd.read_csv(TARGET_PATH, encoding='utf-8')

df

,출발항코드(DEPARTURE_PORT_CODE),출발항명(DEPARTURE_PORT_NAME),도착항코드(DEST_PORT_CODE),도착항명(DEST_PORT_NAME),도착항국가(DEST_COUNTRY),기준일자(DATE),항만효율성(PORT_EFFICIENCY),총항해시간(TOTAL_SAILING_TIME),대기시간(WAITING_TIME),항만정시성(ON_TIME_PERFORMANCE)
0,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-05-28,23.69,644.50,0.00,7.94
1,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-11,22.01,608.54,0.00,16.20
2,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-11,16.52,749.45,0.00,1.36
3,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-30,22.37,634.28,0.00,7.44
4,KRPUS,부산항,AEJEA,제벨알리항,AE,2025-06-30,16.37,654.99,0.00,1.80
...,...,...,...,...,...,...,...,...,...,...
475,KRPUS,부산항,VNSGN,호치민항,VN,2026-05-29,17.90,158.42,0.16,11.23
476,KRPUS,부산항,VNSGN,호치민항,VN,2026-06-15,17.12,216.15,0.18,31.91
477,KRPUS,부산항,VNSGN,호치민항,VN,2026-06-15,17.32,237.27,0.05,7.17
478,KRPUS,부산항,VNSGN,호치민항,VN,2026-06-15,18.19,232.12,0.02,22.30


In [20]:
df_copy = df.copy()

# 컬럼명 간단하게 변경
df_copy = df_copy.rename(columns={
    '기준일자(DATE)': '기준일자', '항만효율성(PORT_EFFICIENCY)': '항만효율성', '총항해시간(TOTAL_SAILING_TIME)': '총항해시간', '대기시간(WAITING_TIME)': '대기시간', '항만정시성(ON_TIME_PERFORMANCE)': '항만정시성'})

df_copy['기준일자'] = pd.to_datetime(df_copy['기준일자'])

In [21]:
# 종합
total_df = (
    df_copy[df_copy['도착항코드(DEST_PORT_CODE)'] == 'KPLI'][['기준일자', '항만효율성', '총항해시간', '대기시간', '항만정시성']]
    .rename(columns={'항만효율성': '종합항만효율성', '총항해시간': '종합총항해시간', '대기시간': '종합대기시간', '항만정시성': '종합항만정시성'}).copy()
)

# 개별 노선
route_df = df_copy[df_copy['도착항코드(DEST_PORT_CODE)'] != 'KPLI'].copy()

In [22]:
daily_avg = (
    route_df.groupby('기준일자')
    .agg(평균_항만효율성=('항만효율성', 'mean'), 평균_총항해시간=('총항해시간', 'mean'), 평균_대기시간=('대기시간', 'mean'), 평균_항만정시성=('항만정시성', 'mean')).reset_index()
)

In [23]:
compare = pd.merge(
    daily_avg,
    total_df,
    on='기준일자',
    how='inner'
)

compare

,기준일자,평균_항만효율성,평균_총항해시간,평균_대기시간,평균_항만정시성,종합항만효율성,종합총항해시간,종합대기시간,종합항만정시성
0,2025-05-28,18.211111,341.750000,0.123333,3.035556,18.99,239.37,0.13,2.29
1,2025-06-11,20.122500,406.270000,0.025833,13.426667,21.48,280.38,0.06,10.84
2,2025-06-11,20.122500,406.270000,0.025833,13.426667,20.45,446.38,0.00,23.58
3,2025-06-30,21.222593,510.192222,0.112593,12.230741,20.06,324.43,0.19,5.72
4,2025-06-30,21.222593,510.192222,0.112593,12.230741,20.67,319.09,0.11,9.72
5,2025-06-30,21.222593,510.192222,0.112593,12.230741,19.70,299.25,0.07,13.01
6,2025-07-08,22.956000,562.422000,0.137000,-5.026000,21.16,361.97,0.12,4.02
7,2025-07-16,18.936667,269.520000,0.150000,-4.938333,19.06,251.70,0.11,0.90
8,2025-07-29,21.402500,427.820000,0.127500,6.186875,18.63,232.41,0.15,3.09
9,2025-07-29,21.402500,427.820000,0.127500,6.186875,19.45,326.72,0.07,7.87
